# FitSync: train the Indian food detector (IndianFoodNet)

This notebook trains YOLOv8 on **IndianFoodNet** (30 Indian dishes) on a free Colab GPU. It then compares the result with the published baseline: **YOLOv8, mAP@50 = 0.807**, from Agarwal et al., *IndianFoodNet: Detecting Indian Food Items Using Deep Learning*.

**Before you start:**
1. Go to *Runtime → Change runtime type → T4 GPU*.
2. Download the dataset from https://universe.roboflow.com/indianfoodnet/indianfoodnet. Choose **Download Dataset → YOLOv8 → download zip**.
3. Choose *Runtime → Run all*. When the upload box appears, pick the zip.

Training takes about 30–45 min. At the end, two files download automatically. Put them in your `foodvol` project:
- `indian_food_yolov8.pt` goes in `foodvol/models/`
- `indian_detector_metrics.json` goes in `foodvol/results/`

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics

In [ ]:
from google.colab import files
up = files.upload()            # choose the Roboflow YOLOv8 zip
ZIP = list(up.keys())[0]
print(ZIP)

In [ ]:
import zipfile, yaml, pathlib
root = pathlib.Path('/content/indianfoodnet'); root.mkdir(exist_ok=True)
zipfile.ZipFile(ZIP).extractall(root)
yml = next(root.rglob('data.yaml')); base = yml.parent
d = yaml.safe_load(open(yml))
for split, folder in (('train','train'),('val','valid'),('test','test')):
    p = base/folder/'images'
    if p.exists(): d[split] = str(p)
    elif split == 'test': d.pop('test', None)
d.pop('path', None)
DATA = str(base/'data_fixed.yaml'); yaml.safe_dump(d, open(DATA,'w'), sort_keys=False)
print(len(d['names']), 'classes:', d['names'])
for f in ('train','valid','test'):
    p = base/f/'images'
    print(f, len(list(p.glob('*'))) if p.exists() else 0, 'images')

## Train
YOLOv8s at 640 px for 60 epochs, with early stopping, cosine learning rate and mosaic + mixup augmentation. The paper trained for only 5 epochs.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data=DATA, epochs=60, imgsz=640, batch=16, patience=20, cos_lr=True,
            close_mosaic=10, mixup=0.1, project='/content/runs', name='indian_det', exist_ok=True)

## Evaluate on the held-out test split and compare with the paper

In [ ]:
import json, shutil
best = model.trainer.best
split = 'test' if 'test' in d else 'val'
v = YOLO(best).val(data=DATA, split=split, imgsz=640)
PAPER = {'model':'YOLOv8 (IndianFoodNet paper, 5 epochs)','precision':0.775,'recall':0.719,'map50':0.807,'map5095':0.628}
ours = {'model':'YOLOv8s (60 epochs, ours)','precision':float(v.box.mp),'recall':float(v.box.mr),
        'map50':float(v.box.map50),'map5095':float(v.box.map)}
names = d['names']
per_class = {(names[i] if isinstance(names, list) else names[i]): float(x) for i, x in zip(v.box.ap_class_index, v.box.ap50)}
res = {'dataset':'IndianFoodNet (Roboflow)','split':split,'n_classes':len(names),'paper':PAPER,'ours':ours,'per_class_ap50':per_class}
json.dump(res, open('/content/indian_detector_metrics.json','w'), indent=1)
shutil.copy(best, '/content/indian_food_yolov8.pt')
print(f"Paper YOLOv8 : mAP50={PAPER['map50']:.3f}  mAP50-95={PAPER['map5095']:.3f}")
print(f"Ours         : mAP50={ours['map50']:.3f}  mAP50-95={ours['map5095']:.3f}  ({split} split)")

In [ ]:
from IPython.display import Image, display
for f in ('results.png','confusion_matrix_normalized.png','val_batch0_pred.jpg'):
    p = pathlib.Path('/content/runs/indian_det')/f
    if p.exists(): display(Image(str(p), width=900))

In [ ]:
files.download('/content/indian_food_yolov8.pt')
files.download('/content/indian_detector_metrics.json')